# Solution of  Linear Systems of Equations

The HHL algorithm (based on surnames Harrow-Hassidim-Lloyd) - quantum algorithm to solve systems of linear equations 
$$A \vec{x} = \vec{b}$$.

To execute this algorithm, we require 4 main steps with three qubit registers:
<br><b>Step 1 :</b> Express the vector $\vec{b}$ as a quantum state $|b\rangle$ on a quantum register
<br><b>Step 2 :</b> Decompose $\vec{b}$ into a superposition of eigenvectors of $A$ remembering on the linear combination of the vector $\vec{b}$. We achieve this using the Quantum Phase Estimation algorithm (Quantum Phase Estimation (QPE)). Since the matrix is hereby diagonalized wherefore $A$ is easily invertible.
<br><b>Step 3 :</b> The inversion of the eigenvector base of $A$ is achieved by rotating an ancillary qubit by an angle $\arcsin \left( \frac{C}{\lambda _{\text{i}}} \right)$ around the y-axis where $\lambda_{\text{i}}$ are the eigenvalues of $A$. Now, we obtain the state $A^{-1}|b\rangle = |x \rangle$.
<br><b>Step 4 :</b> We need to uncompute the register storing the eigenvalues using the inverse QPE. We measure the ancillary qubit whereby the measurement of 1 indicates that the matrix inversion was successful. The inverse QPE leaves the system in a state proportional to the solution vector $|x\rangle$. In many cases one is not interested in the single vector elements of $|x\rangle$ but only on certain properties. These are accessible by applying a problem-specific operator $M$ to the state $|x\rangle$. 

Another use-case of the HHL algorithm is the implementation in a larger quantum program. 
</ol> Currently only hermitian matrices with a dimension of $2^n$ are supported.

Take into account that in the general case, the entries of $\vec{x}$ can not be efficiently read out because we would need to know all coefficients describing the quantum state. In the following examples, we ignore this constraint and show for our small linear system as a proof of principle that $\vec{x}$ is calculated correctly.

In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
import numpy as np
from qiskit_backend_config import BasicAer, execute
# Qiskit Aqua (HHL, EigsQPE, LookupRotation, ...) was removed, and HHL is not part of Qiskit 2.x or
# qiskit-algorithms. The next cell builds HHL from basic circuit parts with the same steps.


In [ ]:
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit.library import QFTGate, RYGate, StatePreparation, UnitaryGate
from qiskit.quantum_info import Statevector, state_fidelity


def matrix_resize(matrix, vector):
    """Pad to a power-of-two size and make the system Hermitian (as Aqua's HHL.matrix_resize did)."""
    matrix, vector = np.asarray(matrix, dtype=complex), np.asarray(vector, dtype=complex)
    truncate_hermitian = not np.allclose(matrix, matrix.conj().T)
    if truncate_hermitian:   # [[0, A], [A^dagger, 0]] (x', x) = (b, 0)
        n = len(matrix)
        matrix = np.block([[np.zeros((n, n)), matrix], [matrix.conj().T, np.zeros((n, n))]])
        vector = np.concatenate([vector, np.zeros(n)])
    size = 1 << int(np.ceil(np.log2(len(matrix))))
    truncate_powerdim = size != len(matrix)
    if truncate_powerdim:
        pad = size - len(matrix)
        matrix = np.block([[matrix, np.zeros((len(matrix), pad))], [np.zeros((pad, len(matrix))), np.eye(pad)]])
        vector = np.concatenate([vector, np.zeros(pad)])
    return matrix, vector, truncate_powerdim, truncate_hermitian


class HHL:
    """Harrow-Hassidim-Lloyd linear solver built from basic circuit parts.

    Registers: |b> (log2 N qubits), clock (num_ancillae [+1 sign bit]) and one rotation ancilla.
    1. prepare |b>   2. QPE of exp(iAt)   3. controlled rotations RY(2 arcsin(C/lambda))
    4. uncompute the QPE; post-select ancilla = 1, clock = 0 -> |x> ~ A^-1 |b>
    """

    def __init__(self, matrix, vector, num_ancillae=3, negative_evals=False, scale=1.0, orig_size=None):
        self._matrix = np.asarray(matrix, dtype=complex)
        self._vector = np.asarray(vector, dtype=complex)
        self._orig_size = orig_size or len(vector)
        self._negative_evals = negative_evals
        self._num_clock = num_ancillae + (1 if negative_evals else 0)
        # evolution time: put the smallest |eigenvalue| exactly on the QPE grid if the largest
        # one still fits in the clock register, otherwise map the largest one to the top value
        evals = np.abs(np.linalg.eigvalsh(self._matrix))
        lambda_min, lambda_max = evals.min(), evals.max()
        top = 2 ** (self._num_clock - (1 if negative_evals else 0)) - 1   # largest magnitude index
        j_min = int(lambda_min * top / lambda_max + 1e-7)
        if j_min >= 1:
            self._evo_time = 2 * np.pi * j_min / (2 ** self._num_clock * lambda_min)
        else:
            self._evo_time = 2 * np.pi * top / (2 ** self._num_clock * lambda_max)
        self._lambda_res = 2 * np.pi / (self._evo_time * 2 ** self._num_clock)
        # constant C in C / lambda: the smallest eigenvalue the clock register can represent
        self._c = scale * self._lambda_res * max(1, round(lambda_min / self._lambda_res))
        self.circuit = self._construct_circuit()

    def _construct_circuit(self):
        nb = int(np.log2(len(self._matrix)))
        b = QuantumRegister(nb, 'b')
        clock = QuantumRegister(self._num_clock, 'clock')
        anc = QuantumRegister(1, 'anc')
        qc = QuantumCircuit(b, clock, anc, name='HHL')
        qc.append(StatePreparation(self._vector / np.linalg.norm(self._vector)), b)

        qpe = QuantumCircuit(b, clock, name='QPE')
        qpe.h(clock)
        for k in range(self._num_clock):
            u_k = UnitaryGate(expm(1j * self._matrix * self._evo_time * 2 ** k), label=f'U^{2 ** k}')
            qpe.append(u_k.control(1), [clock[k]] + list(b))
        qpe.append(QFTGate(self._num_clock).inverse(), clock)
        qc.compose(qpe, qubits=list(b) + list(clock), inplace=True)

        for j in range(1, 2 ** self._num_clock):    # eigenvalue inversion
            signed = j - 2 ** self._num_clock if self._negative_evals and j >= 2 ** (self._num_clock - 1) else j
            angle = 2 * np.arcsin(np.clip(self._c / (signed * self._lambda_res), -1, 1))
            qc.append(RYGate(angle).control(self._num_clock, ctrl_state=j), list(clock) + [anc[0]])

        qc.compose(qpe.inverse(), qubits=list(b) + list(clock), inplace=True)
        return qc

    def postselected_vector(self, backend=None):
        """Amplitudes of |b> when the ancilla is 1 and the clock register is back to 0."""
        nb = self.circuit.qregs[0].size
        if backend is None:
            sv = Statevector(self.circuit).data
        else:   # a statevector simulator backend
            sv = np.asarray(execute(self.circuit, backend).result().get_statevector())
        offset = 1 << (nb + self._num_clock)
        return sv[offset: offset + 2 ** nb]

    def run(self, backend=None):
        vec = self.postselected_vector(backend)
        probability = float(np.real(np.vdot(vec, vec)))
        vec = vec / np.linalg.norm(vec)
        # rescale like Aqua: match |A x| to |b| and fix the global phase
        tmp = self._matrix @ vec
        f1 = np.linalg.norm(self._vector) / np.linalg.norm(tmp)
        nonzero = np.abs(self._vector) > 1e-12
        f2 = np.mean(np.angle(self._vector[nonzero] * tmp[nonzero].conj()))
        solution = (f1 * vec * np.exp(-1j * f2))[: self._orig_size]
        basis = transpile(self.circuit, basis_gates=['u', 'cx'], optimization_level=0)
        return {'solution': solution, 'probability_result': probability,
                'circuit_info': {'width': basis.num_qubits, 'depth': basis.depth()}}


def fidelity(hhl, ref):
    solution_hhl_normed = hhl / np.linalg.norm(hhl)
    solution_ref_normed = ref / np.linalg.norm(ref)
    fidelity = state_fidelity(solution_hhl_normed, solution_ref_normed)
    print("fidelity %f" % fidelity)


class ExactLSsolver:
    """Classical reference solution with numpy (replaces Aqua's ExactLSsolver)."""
    def __init__(self, matrix, vector):
        self._matrix, self._vector = np.asarray(matrix), np.asarray(vector)

    def run(self):
        return {'solution': np.linalg.solve(self._matrix, self._vector)}


In [ ]:
#2x2 diagonal matrix
#First, we show an example for running the HHL algorithm with Qiskit Aqua on a diagonal matrix as input$$
#A=
#\begin{bmatrix}
#1 &amp; 0 \\
#0 &amp; 2
#\end{bmatrix}$$with the vector$$\vec{b}= \left( \begin{array}{c}1 \\ 4  \end{array} \right)$$The result dictionary contains several return values. The HHL solution for $\vec{x}$ is accessible by the key 'solution'. For comparison, also the classical solution of the linear system of equations is calculated using standard linear algebra functions in numpy. The fidelity between the HHL solution and the classical solution is also given in the output. Furthermore, the probability is shown with which HHL was running successfully, i.e. the HHL ancillary qubit has been measured to be $|1\rangle$.
matrix = [[1, 0], [0, 2]]
vector = [1, 4]
orig_size = len(vector)
matrix, vector, truncate_powerdim, truncate_hermitian = matrix_resize(matrix, vector)

# HHL with 3 eigenvalue (clock) qubits
algo = HHL(matrix, vector, num_ancillae=3, negative_evals=False, orig_size=orig_size)

In [ ]:
result = algo.run(BasicAer.get_backend('statevector_simulator'))
print("solution ", np.round(result['solution'], 5))

In [ ]:
result_ref = ExactLSsolver(matrix, vector).run()
print("classical solution ", np.round(result_ref['solution'], 5))
print("probability %f" % result['probability_result'])
fidelity(result['solution'], result_ref['solution'])

In [ ]:
#The probabilty that HHL runs successfully 
#depends on the constant $C$ (see step 3. in the introduction). 
#In the HHL algorithm, $C$ can be given as the parameter scale $\in [0,1]$. 
#In the above example scale is not defined and the HHL algorithm initializes scale 
#to the smallest possible eigenvalue before execution. 
#Alternatively, we can set scale to 0.5 and see how the results are influenced thereby.
orig_size = len(vector)
matrix, vector, truncate_powerdim, truncate_hermitian = matrix_resize(matrix, vector)

# HHL with 3 eigenvalue (clock) qubits
algo = HHL(matrix, vector, num_ancillae=3, negative_evals=False, scale=0.5, orig_size=orig_size)

In [ ]:
result = algo.run(BasicAer.get_backend('statevector_simulator'))
print("solution ", np.round(result['solution'], 5))

result_ref = ExactLSsolver(matrix, vector).run()
print("classical solution ", np.round(result_ref['solution'], 5))

print("probability %f" % result['probability_result'])
fidelity(result['solution'], result_ref['solution'])

In [ ]:
#If you want to know how many qubits are required (circuit width) or 
#how large the maximum number of gates applied to a single qubit (circuit depth) is, 
#you can print it out by
print("circuit_width", result['circuit_info']['width'])
print("circuit_depth", result['circuit_info']['depth'])

In [ ]:
#2x2 non-diagonal matrix
#Here we show an example for running the HHL algorithm with Qiskit Aqua on a non-diagonal matrix 
#as input$$
#A=
#\begin{bmatrix}
#1 &amp; 3 \\
#3 &amp; 2
#\end{bmatrix}$$with the vector$$\vec{b}= \left( \begin{array}{c}1 \\ 1  \end{array} \right)$$
matrix = [[1, 3], [3, 2]]
vector = [1, 1]

orig_size = len(vector)
matrix, vector, truncate_powerdim, truncate_hermitian = matrix_resize(matrix, vector)

# HHL with 3 eigenvalue (clock) qubits plus a sign qubit for negative eigenvalues
algo = HHL(matrix, vector, num_ancillae=3, negative_evals=True, orig_size=orig_size)

In [ ]:
result = algo.run(BasicAer.get_backend('statevector_simulator'))
print("solution ", np.round(result['solution'], 5))

result_ref = ExactLSsolver(matrix, vector).run()
print("classical solution ", np.round(result_ref['solution'], 5))

print("probability %f" % result['probability_result'])
fidelity(result['solution'], result_ref['solution'])
#Compared to the first example, the circuit depth is increased approximately by a factor of 6
print("circuit_width", result['circuit_info']['width'])
print("circuit_depth", result['circuit_info']['depth'])

In [ ]:
#8x8 non-diagonal matrix
#For simplicity, we show a HHL execution of a linear systom of equations defined by 
#the following 8x8 dimensional matrix$$
#A=
#\begin{bmatrix}
#4 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 1 \\
#0 &amp; 3 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 \\
#0 &amp; 0 &amp; 8 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 \\
#0 &amp; 0 &amp; 0 &amp; 5 &amp; 0 &amp; 0 &amp; 0 &amp; 0 \\
#0 &amp; 0 &amp; 0 &amp; 0 &amp; 2 &amp; 1 &amp; 0 &amp; 0 \\
#0 &amp; 0 &amp; 0 &amp; 0 &amp; 1 &amp; 1 &amp; 0 &amp; 0 \\
#0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 1 &amp; 0 \\
#1 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 0 &amp; 5
#\end{bmatrix}$$and 
#the vector$$\vec{b}= \left( \begin{array}{c}1 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 1 \end{array} \right)$$
matrix = [[4, 0, 0, 0, 0, 0, 0, 1],
          [0, 3, 0, 0, 0, 0, 0, 0],
          [0, 0, 8, 0, 0, 0, 0, 0],
          [0, 0, 0, 5, 0, 0, 0, 0],
          [0, 0, 0, 0, 2, 1, 0, 0],
          [0, 0, 0, 0, 1, 1, 0, 0],
          [0, 0, 0, 0, 0, 0, 1, 0],
          [1, 0, 0, 0, 0, 0, 0, 5]]
vector = [1, 0, 0, 0, 0, 0, 0, 1]

orig_size = len(vector)
matrix, vector, truncate_powerdim, truncate_hermitian = matrix_resize(matrix, vector)

# HHL with 3 eigenvalue (clock) qubits
algo = HHL(matrix, vector, num_ancillae=3, negative_evals=False, orig_size=orig_size)

In [ ]:
result = algo.run(BasicAer.get_backend('statevector_simulator'))
print("solution ", np.round(result['solution'], 5))

result_ref = ExactLSsolver(matrix, vector).run()
print("classical solution ", np.round(result_ref['solution'], 5))

print("probability %f" % result['probability_result'])
fidelity(result['solution'], result_ref['solution'])

#Considering the circuit depth and circuit width
print("circuit_width", result['circuit_info']['width'])
print("circuit_depth", result['circuit_info']['depth'])

In [ ]:
#4x4 randomly-generated matrix
#Now, we show the application of HHL on a randomly-generated 4x4 matrix. 
#We use the function random_hermitian to generate a random hermitian matrix and 
#initialize the random seed to achieve reproducibility of the HHL run. 
#Since the matrix can have negative eigenvalues, 
#the parameters need to be modifies to negative_evals=True in "eigs" and reciprocal, respectively. 
#We choose$$\vec{b}= \left( \begin{array}{c}1 \\ 2 \\ 3 \\ 1 \end{array} \right)$$
from qiskit.quantum_info import random_hermitian
#In this example, we create an instance of the HHL class and run the algorithm with an input that is created programatically. To get the same pseudo-random matrix for every run, we set the random seed by using np.random.seed(1).

# set the random seed to get the same pseudo-random matrix for every run
np.random.seed(1)
matrix = random_hermitian(4, seed=1).data
vector = [1, 2, 3, 1]

print("random matrix:")
m = np.array(matrix)
print(np.round(m, 3))

orig_size = len(vector)
matrix, vector, truncate_powerdim, truncate_hermitian = matrix_resize(matrix, vector)

# HHL with 6 eigenvalue (clock) qubits plus a sign qubit for negative eigenvalues
algo = HHL(matrix, vector, num_ancillae=6, negative_evals=True, orig_size=orig_size)

In [ ]:
result = algo.run(BasicAer.get_backend('statevector_simulator'))

print("solution ", np.round(result['solution'], 5))

result_ref = ExactLSsolver(matrix, vector).run()
print("classical solution ", np.round(result_ref['solution'], 5))

print("probability %f" % result['probability_result'])
fidelity(result['solution'], result_ref['solution'])

#The circuit width and depth are
print("circuit_width", result['circuit_info']['width'])
print("circuit_depth", result['circuit_info']['depth'])

# References
1. A. W. Harrow, A. Hassidim, and S. Lloyd, Phys. Rev. Lett. 103, 150502 (2009), e-print arXiv 0811.3171<br>
2. S. Barz, I. Kassal, M. Ringbauer, Y. Ole Lipp, B. Dakić, A. Aspuru-Guzik, and P. Walther, Sci Rep. 4: 6115 (2014), e-print arXiv 1302.1210<br>